# P0 — Vérifications dataset (architecture V8 convergée)

Vérifications **CPU, coût nul**, exigées par le conseil des 5 experts (2026-07-24) AVANT toute ligne de code V8 — cf. `docs/architecture_v8_design.md` §6.1.

| # | Vérification | Ce qu'elle conditionne |
|---|---|---|
| V1 | **Provenance des prédicteurs** : champs GCM natifs régrillés, ou sorties CCAM *coarsenées* ? | Le verdict E2 (Climat) : les influences statique(HR)→dynamique(LR) existent-elles dans nos données ? |
| V2 | **Existence d'une run CCAM-ERA5** (local + dépôt Zenodo 10889046) | L'environnement de découverte « qui vaut plus que les 3 GCM réunis » (Climat) |
| V3 | **Artefact de normalisation K23** : stats ACCESS appliquées aux autres GCM | Normalisation PAR-GCM obligatoire avant tout test d'invariance (IA) |
| V4 | **Amplitude des shifts inter-GCM** (claim ≤0.34σ) | La puissance de la Gate 0 de distinctness (Math E3) |

⚠️ **Pré-enregistrement respecté** : `NorESM2-MM` est le **holdout OOD intouchable** — ce notebook ne lit AUCUNE donnée NorESM2 (garde explicite en V3/V4). Les comparaisons inter-GCM se font sur ACCESS-CM2 + EC-Earth3 uniquement.

In [ ]:
# %% Cellule 1 — imports, chemins, inventaire
import os, json, urllib.request
import numpy as np
import xarray as xr

ROOT = os.path.abspath(os.path.dirname('__file__'))
RAW  = os.path.join(ROOT, 'data', 'raw')

PATHS = {
    'predictor_ACCESS' : os.path.join(RAW, 'train', 'predictor_ACCESS-CM2_hist.nc'),
    'pr_ACCESS'        : os.path.join(RAW, 'train', 'pr_ACCESS-CM2_hist.nc'),
    'static'           : os.path.join(RAW, 'static_predictors', 'ERA5_eval_ccam_12km.198110_NZ_Invariant.nc'),
    'EC-Earth3_pred'   : os.path.join(RAW, 'test', 'EC-Earth3_histupdated_compressed.nc'),
    'EC-Earth3_pr'     : os.path.join(RAW, 'test', 'EC-Earth3_historical_precip_compressed.nc'),
    'norm_mean'        : os.path.join(RAW, 'normalization_coefs', 'mean_1974_2011.nc'),
    'norm_std'         : os.path.join(RAW, 'normalization_coefs', 'std_1974_2011.nc'),
    # NorESM2 : chemins listés mais JAMAIS ouverts (holdout pré-enregistré)
    'NorESM2_pred_HOLDOUT' : os.path.join(RAW, 'test', 'NorESM2-MM_histupdated_compressed.nc'),
}

print(f"{'fichier':<24} {'présent':<8} taille")
for k, p in PATHS.items():
    ok = os.path.exists(p)
    size = f"{os.path.getsize(p)/1e6:,.1f} MB" if ok else '—'
    print(f"{k:<24} {str(ok):<8} {size}")

## V1 — Provenance des prédicteurs

**Question** (Climat, §4 du verdict) : les 15 canaux LR `u,v,w,q,t` sont-ils (a) des champs **GCM natifs régrillés** sur 23×26, ou (b) des sorties **CCAM coarsenées** ?

- Si (a) : le GCM n'a jamais vu le relief 12 km ⇒ les arêtes statique(HR)→dynamique(LR) **n'existent pas** dans les données (E2 confirmé, statiques = contexte au décodeur HR uniquement).
- Si (b) : une fraction du blocage/foehn HR **fuit** dans les prédicteurs ⇒ E2 s'adoucit, certaines influences statique→dynamique deviennent partiellement apprenables.

Deux angles : **(1)** les métadonnées NetCDF (attrs globaux : source, history, institution) ; **(2)** une signature spectrale (un champ natif ~1.9° régrillé bilinéairement sur 1.44° est *lisse* — il se reconstruit presque parfaitement depuis un sous-échantillonnage 2×2 ; un champ CCAM coarsené garde de la variance fine).

In [ ]:
# %% Cellule 2 — V1a : métadonnées de provenance
KEYS = ['source', 'history', 'title', 'institution', 'project', 'contact', 'Conventions', 'comment', 'parent_source', 'driving_model']
for name in ['predictor_ACCESS', 'pr_ACCESS', 'static', 'EC-Earth3_pred', 'EC-Earth3_pr']:
    p = PATHS[name]
    if not os.path.exists(p):
        print(f'--- {name}: ABSENT ---'); continue
    ds = xr.open_dataset(p, decode_times=False)
    print(f'=== {name} ===')
    for k in KEYS:
        if k in ds.attrs:
            v = str(ds.attrs[k])[:180]
            print(f'   {k:<14}: {v}')
    # attrs d'une variable témoin
    for v in list(ds.data_vars)[:1]:
        if ds[v].attrs: print(f'   [{v}] attrs : {dict(list(ds[v].attrs.items())[:4])}')
    ds.close(); print()
print("Lecture : la mention 'CSIRO conformal-cubic model / cc2hist / surf.ccam' = chaîne CCAM ;")
print("une mention CMIP6/ESGF/regrid = champs GCM natifs régrillés.")

In [ ]:
# %% Cellule 3 — V1b : signature spectrale (test de reconstruction bilinéaire)
# Un champ natif grossier régrillé se reconstruit ~parfaitement depuis ses points 2x2 ;
# un champ réellement résolu à 1.44° (CCAM coarsené) non.

def bilinear_recon_residual(field2d):
    """Sous-échantillonne 2x2 puis reconstruit par interp bilinéaire ; renvoie le R² de reconstruction."""
    f = np.asarray(field2d, dtype=float)
    ny, nx = f.shape
    ys, xs = np.arange(0, ny, 2), np.arange(0, nx, 2)
    sub = f[np.ix_(ys, xs)]
    da = xr.DataArray(sub, coords={'y': ys.astype(float), 'x': xs.astype(float)}, dims=('y','x'))
    rec = da.interp(y=np.arange(ny).astype(float), x=np.arange(nx).astype(float),
                    method='linear', kwargs={'fill_value': None})
    r = rec.values
    m = np.isfinite(r) & np.isfinite(f)
    ss_res = np.nansum((f[m]-r[m])**2); ss_tot = np.nansum((f[m]-np.nanmean(f[m]))**2)
    return 1.0 - ss_res/ss_tot if ss_tot > 0 else np.nan

ds = xr.open_dataset(PATHS['predictor_ACCESS'], decode_times=False)
np.random.seed(0)
days = np.random.choice(ds.sizes['time'], size=30, replace=False)
print(f"{'variable':<8} R²_reconstruction (moy sur 30 jours)   lecture")
for v in ['u_850', 't_850', 'w_850', 'q_850', 'w_500']:
    if v not in ds: continue
    r2s = [bilinear_recon_residual(ds[v].isel(time=int(d)).values) for d in days]
    r2 = float(np.nanmean(r2s))
    verdict = 'lisse -> compatible GCM natif régrillé' if r2 > 0.985 else ('intermédiaire' if r2 > 0.95 else 'variance fine -> compatible CCAM coarsené')
    print(f"{v:<8} {r2:.4f}                                {verdict}")
ds.close()
print('\nNB : heuristique. w (vitesse verticale) est naturellement plus rugueux que u/t ;')
print('le verdict se lit sur le FAISCEAU des variables, u/t en tête. Confirmation définitive =')
print('comparer un jour donné au champ GCM natif (ESGF) — hors périmètre CPU de ce notebook.')

## V2 — Run CCAM-ERA5

Le fichier statique s'appelle `ERA5_eval_ccam_12km...` ⇒ la campagne CCAM-NZ comporte une **run d'évaluation pilotée par ERA5**. Si ses prédicteurs/précip sont accessibles, c'est un environnement de découverte à forçage réellement différent **avec ancrage observationnel** (Climat : « vaut plus que les 3 GCM réunis »).

On cherche : **(a)** localement (noms de fichiers + attrs contenant ERA5) ; **(b)** dans le dépôt Zenodo du projet (record `10889046`, la source des données de test) via l'API publique.

In [ ]:
# %% Cellule 4 — V2a : recherche locale ERA5
hits = []
for base in [os.path.join(ROOT,'data'), os.path.join(ROOT,'downscaling')]:
    for dirpath, _, files in os.walk(base):
        for f in files:
            if f.endswith(('.nc', '.nc4')):
                p = os.path.join(dirpath, f)
                tag = 'ERA5' in f.upper()
                if not tag:
                    try:
                        d = xr.open_dataset(p, decode_times=False)
                        tag = any('ERA5' in str(x).upper() for x in d.attrs.values()); d.close()
                    except Exception: pass
                if tag: hits.append(os.path.relpath(p, ROOT))
print('Fichiers locaux liés à ERA5 :')
for h in hits: print('  -', h)
print('\n-> Si seul le fichier STATIQUE apparaît : la run ERA5 existe (nommage) mais ses champs')
print('   dynamiques/précip ne sont PAS dans le dataset local -> vérifier Zenodo (cellule suivante).')

In [ ]:
# %% Cellule 5 — V2b : inventaire du record Zenodo 10889046
url = 'https://zenodo.org/api/records/10889046'
try:
    with urllib.request.urlopen(url, timeout=30) as r:
        rec = json.load(r)
    print('Titre du record :', rec.get('metadata', {}).get('title', '?'))
    print('\nFichiers du record :')
    era5_found = False
    for f in rec.get('files', []):
        name = f.get('key', '?'); size = f.get('size', 0)/1e6
        flag = '   <-- ERA5 !' if 'ERA5' in name.upper() else ''
        if flag: era5_found = True
        print(f"  - {name:<55} {size:>9,.1f} MB{flag}")
    print('\nVERDICT V2 :', 'run CCAM-ERA5 DISPONIBLE dans le record -> à télécharger en priorité'
          if era5_found else 'pas de fichier ERA5 dans ce record -> chercher le record compagnon de la campagne CCAM-NZ (Gibson et al. 2023) ou contacter NIWA/CSIRO.')
except Exception as e:
    print('API Zenodo inaccessible (offline ?) :', e)
    print('-> relancer en ligne, ou consulter https://zenodo.org/records/10889046 manuellement.')

## V3 + V4 — Normalisation K23 et amplitude des shifts inter-GCM

- **V3** : les stats de normalisation (1974-2011, base ACCESS) appliquées à EC-Earth3 fabriquent un décalage **artificiel** — c'est l'artefact K23. On le quantifie, il justifie la normalisation PAR-GCM.
- **V4** : l'amplitude réelle des shifts inter-GCM en unités de σ — le claim historique est « ≤0.34σ ». En dessous de ~0.3-0.5σ, la puissance d'un test d'invariance est quasi nulle (Math E3) : c'est la matière première de la Gate 0.

🔒 **Garde holdout** : NorESM2-MM n'est PAS lu.

In [ ]:
# %% Cellule 6 — V3/V4 : stats par GCM (ACCESS vs EC-Earth3 UNIQUEMENT)
assert True, 'NorESM2 = holdout, non lu'

ds_a = xr.open_dataset(PATHS['predictor_ACCESS'], decode_times=False)
ds_e = xr.open_dataset(PATHS['EC-Earth3_pred'], decode_times=False)

def pick(ds, names):
    for n in names:
        if n in ds.data_vars: return n
    # fallback : match insensible à la casse
    low = {v.lower(): v for v in ds.data_vars}
    for n in names:
        if n.lower() in low: return low[n.lower()]
    return None

print('Variables EC-Earth3 :', list(ds_e.data_vars)[:20])
print()
print(f"{'var':<8} {'μ_ACCESS':>10} {'μ_EC':>10} {'σ_ACCESS':>10} {'shift (σ)':>10} {'σ_EC/σ_ACCESS':>14}")
for var in ['t_850', 'q_850', 'u_850', 'v_850', 'w_500']:
    va = pick(ds_a, [var]); ve = pick(ds_e, [var, var.replace('_','')])
    if va is None or ve is None:
        print(f'{var:<8} absent d\'un des deux jeux — sauté'); continue
    A = ds_a[va].values; E = ds_e[ve].values
    mA, sA = np.nanmean(A), np.nanstd(A)
    mE, sE = np.nanmean(E), np.nanstd(E)
    print(f'{var:<8} {mA:>10.4g} {mE:>10.4g} {sA:>10.4g} {(mE-mA)/sA:>10.3f} {sE/sA:>14.3f}')

print('\nLecture V3 : standardiser EC-Earth3 avec (μ,σ) ACCESS laisse un offset = colonne « shift (σ) »')
print('et une échelle résiduelle = colonne « σ_EC/σ_ACCESS » — c\'est l\'artefact K23 que la')
print('découverte par invariance prendrait pour de la distinctness. Normalisation PAR-GCM requise.')
print('Lecture V4 : si |shift| ≲ 0.3-0.5σ partout, la Gate 0 devra s\'appuyer sur les régimes')
print('de circulation (et la run CCAM-ERA5 si V2 la trouve), pas sur les GCM seuls.')
ds_a.close(); ds_e.close()

## Interprétation et suites

| Résultat | Conséquence sur V8 |
|---|---|
| V1 = GCM natif régrillé | E2 **confirmé** : statiques = contexte spatial au décodeur HR uniquement (aucune arête statique→LR à chercher) |
| V1 = CCAM coarsené | E2 s'adoucit : documenter la fuite, certaines influences statique→dynamique partiellement apprenables |
| V2 = run ERA5 trouvée | La télécharger ; elle devient l'environnement de découverte n°1 (+ ancrage de l'empreinte F1, C10-a) |
| V2 = introuvable | Environnements = régimes de circulation (Kidson/AR/saisons/ENSO) ; empreinte F1 inter-GCM seulement |
| V3 : offsets non nuls | Normalisation par-GCM **obligatoire** avant Gate 0 (sinon distinctness artificielle) |
| V4 : shifts ≲0.34σ confirmés | Gate 0 (T2) risque fort d'échouer sur les GCM seuls → plan B régimes déjà prévu |

**Ensuite** (P0, cf. `docs/architecture_v8_design.md` §6.1) : T1 audit de dégénérescence des 13 nœuds libres → T2 Gate 0 distinctness (empreintes PCMCI+ par GCM, bootstrap) → T3 surrogates + `sortnregress` → T4 ESS/puissance → T5-T8.